In [ ]:
import cv2 as cv
import numpy as np

cap = cv.VideoCapture(0)

width = cap.get(cv.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv.CAP_PROP_FRAME_HEIGHT)
fps = 60.0 

filtro_aplicado = None
kernel_aplicado = 3
ruido_aplicado = None

def aplicar_ruido(img, tipo):
    if tipo == "gaussiano":
        row, col, ch = img.shape
        mean = 0
        var = 100 
        sigma = var**0.5
        gauss = np.random.normal(mean, sigma, (row, col, ch))
        noisy = np.clip(img.astype(np.float32) + gauss, 0, 255).astype(np.uint8)
        return noisy
        
    elif tipo == "sal-pimenta":
        noisy = np.copy(img)
        prob = 0.05
        rdn = np.random.random(img.shape[:2])
        noisy[rdn < prob/2] = [0, 0, 0]
        noisy[rdn > 1 - prob/2] = [255, 255, 255]
        return noisy
        
    return img

def validar_filtro_selecionado(tecla_pressionada):
    if tecla_pressionada == ord('a'):
        return "media"
    elif tecla_pressionada == ord('g'):
        return "gaussiano"
    elif tecla_pressionada == ord('m'):
        return "mediana"
    elif tecla_pressionada == ord('b'):
        return "bilateral"
    return None

def validar_kernel_selecionado(tecla_pressionada):
    if tecla_pressionada == ord('3'):
        return 3
    elif tecla_pressionada == ord('5'):
        return 5
    elif tecla_pressionada == ord('7'):
        return 11
    elif tecla_pressionada == ord('9'):
        return 29
    elif tecla_pressionada == ord('x'):
        return 47
    return None

def validar_ruido_selecionado(tecla_pressionada):
    if tecla_pressionada == ord('o'):
        return None
    elif tecla_pressionada == ord('y'):
        return "gaussiano"
    elif tecla_pressionada == ord('p'):
        return "sal-pimenta"
    return None

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    frame_com_ruido = aplicar_ruido(frame, ruido_aplicado)
    frame_filtrado = frame_com_ruido.copy()

    if filtro_aplicado == "media":
        frame_filtrado = cv.blur(frame_com_ruido, (kernel_aplicado, kernel_aplicado))
    elif filtro_aplicado == "gaussiano":
        frame_filtrado = cv.GaussianBlur(frame_com_ruido, (kernel_aplicado, kernel_aplicado), 0)
    elif filtro_aplicado == "mediana":
        frame_filtrado = cv.medianBlur(frame_com_ruido, kernel_aplicado)
    elif filtro_aplicado == "bilateral":
        frame_filtrado = cv.bilateralFilter(frame_com_ruido, d=kernel_aplicado, sigmaColor=75, sigmaSpace=75)

    cv.imshow('frame', frame_filtrado)
    tecla = cv.waitKey(1) & 0xFF

    if tecla == ord('q'):
        break
    else:
        novo_filtro = validar_filtro_selecionado(tecla)
        if novo_filtro is not None:
            filtro_aplicado = novo_filtro

        novo_kernel = validar_kernel_selecionado(tecla)
        if novo_kernel is not None:
            kernel_aplicado = novo_kernel

        if tecla in [ord('o'), ord('y'), ord('p')]:
            ruido_aplicado = validar_ruido_selecionado(tecla)

cap.release()
cv.destroyAllWindows()